# HW1 — Optical Flow & Video Stabilization (Variants A & B)

This notebook implements **both variants** from the assignment and saves the required artifacts into:
- `outputs/A/` (stabilization)
- `outputs/B/` (optical flow, motion mask, sensitivity)


In [1]:
!pip -q install requests matplotlib opencv-python-headless

In [2]:
import os, math, json, time, subprocess, shutil
from pathlib import Path
import numpy as np
import cv2
import requests
import matplotlib.pyplot as plt

In [3]:
def ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)

def download(urls, out_path, timeout=60):
    out_path = str(out_path)
    ensure_dir(Path(out_path).parent)
    if isinstance(urls, str):
        urls = [urls]
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "*/*"}
    for u in urls:
        try:
            r = requests.get(u, stream=True, timeout=timeout, allow_redirects=True, headers=headers)
            if r.status_code != 200:
                continue
            ct = (r.headers.get("content-type") or "").lower()
            if "text/html" in ct:
                continue
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
            if os.path.getsize(out_path) > 10000:
                with open(out_path, "rb") as f:
                    head = f.read(256).lower()
                if b"<html" in head or b"<!doctype" in head:
                    os.remove(out_path)
                    continue
                return out_path
        except Exception:
            pass
        try:
            cmd = f"curl -L --fail --silent --show-error -A 'Mozilla/5.0' -o '{out_path}' '{u}'"
            p = subprocess.run(["bash", "-lc", cmd], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout + 30)
            if p.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 10000:
                return out_path
        except Exception:
            pass
    return None

def video_meta(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {"fps": float(fps), "frames": int(n), "w": int(w), "h": int(h)}

def affine_to_h(M2x3):
    H = np.eye(3, dtype=np.float32)
    H[:2, :] = M2x3.astype(np.float32)
    return H

def h_to_affine(H):
    return H[:2, :].astype(np.float32)

def apply_h_to_point(H, pt):
    x, y = float(pt[0]), float(pt[1])
    v = np.array([x, y, 1.0], dtype=np.float32)
    w = H @ v
    return np.array([w[0] / w[2], w[1] / w[2]], dtype=np.float32)

def clamp_norm(v, max_norm):
    n = float(np.hypot(v[0], v[1]))
    if n <= max_norm or n <= 1e-9:
        return v
    s = max_norm / n
    return v * s

def odd(k):
    k = int(k)
    return k if (k % 2 == 1) else (k + 1)


In [4]:
from pathlib import Path
import numpy as np

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("data")
OUT_A = Path("outputs/A")
OUT_B = Path("outputs/B")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_A.mkdir(parents=True, exist_ok=True)
OUT_B.mkdir(parents=True, exist_ok=True)


In [5]:
from pathlib import Path
import subprocess

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URL_A = "https://upload.wikimedia.org/wikipedia/commons/3/34/Spider_%28Cyclosa_argenteoalba%29.webm"
URL_B = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"

PATH_A = DATA_DIR / "video_A.webm"
PATH_B = DATA_DIR / "video_B.avi"

def _looks_like_html(p: Path) -> bool:
    b = p.read_bytes()[:128].lstrip()
    return b.startswith(b"<") or b"<!DOCTYPE" in b.upper() or b"<HTML" in b.upper()

def _download(url: str, out_path: Path):
    subprocess.run(
        ["bash", "-lc", f'curl -L --retry 5 --retry-delay 1 -A "Mozilla/5.0" -o "{out_path}" "{url}"'],
        check=True
    )
    if out_path.stat().st_size < 200_000 or _looks_like_html(out_path):
        raise RuntimeError(f"Downloaded file is not a valid video: {out_path}")

if not PATH_A.exists():
    _download(URL_A, PATH_A)

if not PATH_B.exists():
    _download(URL_B, PATH_B)

print("A:", PATH_A, "bytes:", PATH_A.stat().st_size)
print("B:", PATH_B, "bytes:", PATH_B.stat().st_size)

A: data/video_A.webm bytes: 20506674
B: data/video_B.avi bytes: 8131690


In [6]:
def good_features_grid(gray, max_total=800, quality=0.01, min_dist=7, grid=(4, 4)):
    h, w = gray.shape[:2]
    gx, gy = int(grid[0]), int(grid[1])
    per_cell = max(10, int(max_total / (gx * gy)))
    pts = []
    for iy in range(gy):
        for ix in range(gx):
            x0 = int(ix * w / gx)
            x1 = int((ix + 1) * w / gx)
            y0 = int(iy * h / gy)
            y1 = int((iy + 1) * h / gy)
            roi = gray[y0:y1, x0:x1]
            if roi.size == 0:
                continue
            p = cv2.goodFeaturesToTrack(roi, maxCorners=per_cell, qualityLevel=quality, minDistance=min_dist, blockSize=7, useHarrisDetector=False)
            if p is None:
                continue
            p[:, 0, 0] += x0
            p[:, 0, 1] += y0
            pts.append(p)
    if not pts:
        return None
    pts = np.concatenate(pts, axis=0)
    if pts.shape[0] > max_total:
        pts = pts[:max_total]
    return pts.astype(np.float32)

def select_anchor(gray, frac=0.55, max_corners=200):
    h, w = gray.shape[:2]
    cx0 = int((1 - frac) * w / 2)
    cx1 = int(w - cx0)
    cy0 = int((1 - frac) * h / 2)
    cy1 = int(h - cy0)
    roi = gray[cy0:cy1, cx0:cx1]
    p = cv2.goodFeaturesToTrack(roi, maxCorners=max_corners, qualityLevel=0.01, minDistance=7, blockSize=7)
    if p is None:
        return np.array([w / 2, h / 2], dtype=np.float32)
    pts = p.reshape(-1, 2) + np.array([cx0, cy0], dtype=np.float32)
    center = np.array([w / 2, h / 2], dtype=np.float32)
    d = np.sum((pts - center) ** 2, axis=1)
    return pts[np.argmin(d)].astype(np.float32)

def phase_translation(prev_gray, curr_gray):
    a = prev_gray.astype(np.float32)
    b = curr_gray.astype(np.float32)
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(a, b, win)
    if not np.isfinite(dx) or not np.isfinite(dy):
        return np.eye(3, dtype=np.float32), 0.0
    if float(math.hypot(dx, dy)) > 80.0:
        return np.eye(3, dtype=np.float32), float(resp)
    M = np.array([[1.0, 0.0, float(dx)], [0.0, 1.0, float(dy)]], dtype=np.float32)
    return affine_to_h(M), float(resp)

def estimate_motion(prev_gray, curr_gray, pts_prev, ransac_thresh=3.0, fb_thresh=1.0, min_inliers=30):
    lk_params = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    if pts_prev is None or len(pts_prev) < 50:
        pts_prev = good_features_grid(prev_gray, max_total=900)
        if pts_prev is None:
            return np.eye(3, dtype=np.float32), None, {"ok": False, "inliers": 0, "tracked": 0}
    pts_curr, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts_prev, None, **lk_params)
    if pts_curr is None:
        Hph, resp = phase_translation(prev_gray, curr_gray)
        ok = bool(resp >= 0.10)
        return (Hph if ok else np.eye(3, dtype=np.float32)), None, {"ok": ok, "inliers": 0, "tracked": 0}
    pts_back, st2, err2 = cv2.calcOpticalFlowPyrLK(curr_gray, prev_gray, pts_curr, None, **lk_params)
    d = np.linalg.norm(pts_prev.reshape(-1, 2) - pts_back.reshape(-1, 2), axis=1)
    st = st.reshape(-1).astype(bool)
    st2 = st2.reshape(-1).astype(bool)
    good = st & st2 & (d < fb_thresh)
    p0 = pts_prev.reshape(-1, 2)[good]
    p1 = pts_curr.reshape(-1, 2)[good]
    tracked = int(p0.shape[0])
    if tracked < 30:
        Hph, resp = phase_translation(prev_gray, curr_gray)
        ok = bool(resp >= 0.10)
        return (Hph if ok else np.eye(3, dtype=np.float32)), pts_curr.reshape(-1, 1, 2).astype(np.float32), {"ok": ok, "inliers": 0, "tracked": tracked}
    M, inliers = cv2.estimateAffinePartial2D(p0, p1, method=cv2.RANSAC, ransacReprojThreshold=ransac_thresh, maxIters=2000, confidence=0.99, refineIters=10)
    if M is None or inliers is None:
        Hph, resp = phase_translation(prev_gray, curr_gray)
        ok = bool(resp >= 0.10)
        return (Hph if ok else np.eye(3, dtype=np.float32)), pts_curr.reshape(-1, 1, 2).astype(np.float32), {"ok": ok, "inliers": 0, "tracked": tracked}
    inliers = inliers.reshape(-1).astype(bool)
    inl = int(inliers.sum())
    ok = inl >= min_inliers and inl / max(1, tracked) >= 0.25
    sx = float(math.hypot(M[0, 0], M[1, 0]))
    da = float(math.atan2(M[1, 0], M[0, 0]))
    if not (0.85 <= sx <= 1.15):
        ok = False
    if abs(da) > 0.25:
        ok = False
    if not ok:
        Hph, resp = phase_translation(prev_gray, curr_gray)
        ok2 = bool(resp >= 0.12)
        return (Hph if ok2 else np.eye(3, dtype=np.float32)), pts_curr.reshape(-1, 1, 2).astype(np.float32), {"ok": ok2, "inliers": inl, "tracked": tracked}
    return affine_to_h(M), pts_curr.reshape(-1, 1, 2).astype(np.float32), {"ok": True, "inliers": inl, "tracked": tracked}

In [7]:
def smooth_trajectory(x, win=31):
    k = odd(win)
    x = np.asarray(x, dtype=np.float32)
    y = x.reshape(-1, 1)
    ys = cv2.GaussianBlur(y, (k, 1), 0).reshape(-1)
    return ys

def build_smoothed_matrices(rel_H_list, win=31):
    n = len(rel_H_list) + 1
    C = [np.eye(3, dtype=np.float32)]
    for H in rel_H_list:
        C.append(C[-1] @ H)
    C = np.stack(C, axis=0)
    xs = C[:, 0, 2].copy()
    ys = C[:, 1, 2].copy()
    ang = np.arctan2(C[:, 1, 0], C[:, 0, 0]).astype(np.float32)
    xs_s = smooth_trajectory(xs, win=win)
    ys_s = smooth_trajectory(ys, win=win)
    ang_s = smooth_trajectory(ang, win=win)
    S = []
    for i in range(n):
        a = float(ang_s[i])
        ca, sa = float(math.cos(a)), float(math.sin(a))
        H = np.array([[ca, -sa, float(xs_s[i])],
                      [sa,  ca, float(ys_s[i])],
                      [0.0, 0.0, 1.0]], dtype=np.float32)
        S.append(H)
    S = np.stack(S, axis=0)
    return C, S, {"raw": (xs, ys, ang), "smooth": (xs_s, ys_s, ang_s)}

In [8]:
def make_before_after(samples, fps, out_path):
    rows = []
    for idx, a, b in samples:
        t = idx / max(1e-9, fps)
        label = f"frame={idx}  t={t:05.2f}s"
        a2 = a.copy()
        b2 = b.copy()
        cv2.putText(a2, label, (12, a2.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(a2, label, (12, a2.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 1, cv2.LINE_AA)
        cv2.putText(b2, label, (12, b2.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(b2, label, (12, b2.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 1, cv2.LINE_AA)
        pair = np.concatenate([a2, b2], axis=1)
        mid = pair.shape[1] // 2
        pair[:, mid-2:mid+2] = 0
        rows.append(pair)
    grid = np.concatenate(rows, axis=0)
    h, w = grid.shape[:2]
    header_h = max(54, int(0.10 * h / max(1, len(rows))))
    header = np.ones((header_h, w, 3), dtype=np.uint8) * 255
    mid = w // 2
    header[:, mid-2:mid+2] = 0
    fs = 1.1
    th = 2
    for txt, cx in [("INPUT (cropped)", mid//2), ("STABILIZED", mid + (w-mid)//2)]:
        (tw, thh), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
        x = int(cx - tw/2)
        y = int(header_h*0.65)
        cv2.putText(header, txt, (x, y), cv2.FONT_HERSHEY_SIMPLEX, fs, (0,0,0), th, cv2.LINE_AA)
    out = np.concatenate([header, grid], axis=0)
    cv2.imwrite(str(out_path), out)
    return out


def motion_curve_plot(raw, smooth, out_path):
    xs, ys, ang = raw
    xs_s, ys_s, ang_s = smooth
    fig = plt.figure(figsize=(12, 8))
    ax1 = plt.subplot(3,1,1); ax1.plot(xs, label="raw dx"); ax1.plot(xs_s, label="smoothed dx"); ax1.legend()
    ax2 = plt.subplot(3,1,2); ax2.plot(ys, label="raw dy"); ax2.plot(ys_s, label="smoothed dy"); ax2.legend()
    ax3 = plt.subplot(3,1,3); ax3.plot(ang, label="raw angle"); ax3.plot(ang_s, label="smoothed angle"); ax3.legend()
    fig.tight_layout()
    fig.savefig(str(out_path), dpi=160)
    plt.close(fig)

def residual_plot(rel_H_list, warped_H_list, out_path):
    def rel_params(H):
        dx = float(H[0,2]); dy = float(H[1,2])
        da = float(math.atan2(H[1,0], H[0,0]))
        return dx, dy, da
    raw_mag = []
    stab_mag = []
    for i in range(len(rel_H_list)):
        dx, dy, da = rel_params(rel_H_list[i])
        raw_mag.append(math.hypot(dx, dy))
    for i in range(len(warped_H_list)-1):
        A = warped_H_list[i]
        B = warped_H_list[i+1]
        D = np.linalg.inv(A) @ B
        dx, dy, da = rel_params(D)
        stab_mag.append(math.hypot(dx, dy))
    fig = plt.figure(figsize=(12,4))
    plt.plot(raw_mag, label="input")
    plt.plot(stab_mag, label="stabilized")
    plt.legend()
    plt.tight_layout()
    fig.savefig(str(out_path), dpi=160)
    plt.close(fig)

## Variant A — Global Motion Estimation & Video Stabilization

Pipeline (high level): detect & track features → estimate global motion (affine/homography) → smooth camera trajectory → warp frames → auto-crop → save stabilized video + diagnostics.


In [9]:
CFG_A = {
    "ransac_thresh": 3.0,
    "fb_thresh": 1.0,
    "min_inliers": 35,
    "smooth_window": 41,
    "max_anchor_shift": 12.0,
    "anchor_roi_frac": 0.55,
    "crop_percentiles": (5, 95),
    "crop_margin": 6
}

cap = cv2.VideoCapture(str(PATH_A))
if not cap.isOpened():
    raise RuntimeError("Cannot open video A")
fpsA = float(cap.get(cv2.CAP_PROP_FPS))
if not np.isfinite(fpsA) or fpsA <= 1e-3:
    fpsA = 30.0
ret, prev = cap.read()
if not ret:
    raise RuntimeError("Empty video A")
H, W = int(prev.shape[0]), int(prev.shape[1])
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
anchor_ref = select_anchor(prev_gray, frac=CFG_A["anchor_roi_frac"])
anchor_pos = [anchor_ref.copy()]
anchor_ok = [True]
rel_H = []
pts = good_features_grid(prev_gray, max_total=900)
miss = 0

print("Variant A: estimating global motion")
i = 1
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    Hrel, pts_next, st = estimate_motion(prev_gray, gray, pts, ransac_thresh=CFG_A["ransac_thresh"], fb_thresh=CFG_A["fb_thresh"], min_inliers=CFG_A["min_inliers"])
    rel_H.append(Hrel)
    lk_params = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    ap = anchor_pos[-1].reshape(1, 1, 2).astype(np.float32)
    ap2, stA, errA = cv2.calcOpticalFlowPyrLK(prev_gray, gray, ap, None, **lk_params)
    ap_back, stB, errB = cv2.calcOpticalFlowPyrLK(gray, prev_gray, ap2, None, **lk_params)
    fb = float(np.linalg.norm(ap.reshape(2) - ap_back.reshape(2)))
    ok = stA is not None and stB is not None and bool(stA.reshape(-1)[0]) and bool(stB.reshape(-1)[0]) and fb < 2.0
    if ok:
        anchor_pos.append(ap2.reshape(2).astype(np.float32))
        anchor_ok.append(True)
        miss = 0
    else:
        miss += 1
        if miss >= 10:
            anc = select_anchor(gray, frac=CFG_A["anchor_roi_frac"])
            anchor_pos.append(anc.copy())
            anchor_ok.append(True)
            miss = 0
        else:
            anchor_pos.append(anchor_pos[-1].copy())
            anchor_ok.append(False)
    if pts_next is None or len(pts_next) < 120 or (not st["ok"]):
        pts = good_features_grid(gray, max_total=900)
    else:
        pts = pts_next
    prev_gray = gray
    i += 1
cap.release()

n_used = len(rel_H) + 1
anchor_pos = np.stack(anchor_pos[:n_used], axis=0)
anchor_ok = np.array(anchor_ok[:n_used], dtype=bool)
print("Used frames:", n_used, "fps:", fpsA, "anchor ok:", int(anchor_ok.sum()), "size:", (W, H))


Variant A: estimating global motion
Used frames: 980 fps: 23.98 anchor ok: 899 size: (1920, 1280)


In [10]:
C, S, traj = build_smoothed_matrices(rel_H, win=CFG_A["smooth_window"])
W_list = []
ema = np.array([0.0, 0.0], dtype=np.float32)
alpha = 0.15
for i in range(n_used):
    Wi = S[i] @ np.linalg.inv(C[i])
    if anchor_ok[i]:
        pi = apply_h_to_point(Wi, anchor_pos[i])
        corr = (anchor_ref - pi).astype(np.float32)
        corr = clamp_norm(corr, CFG_A["max_anchor_shift"])
        ema = (1 - alpha) * ema + alpha * corr
    else:
        ema = (1 - alpha) * ema
    Ti = np.array([[1.0, 0.0, float(ema[0])],
                   [0.0, 1.0, float(ema[1])],
                   [0.0, 0.0, 1.0]], dtype=np.float32)
    W_list.append((Ti @ Wi).astype(np.float32))
W_list = np.stack(W_list, axis=0)

motion_curve_plot(traj["raw"], traj["smooth"], OUT_A / "camera_trajectory.png")
residual_plot(rel_H, W_list, OUT_A / "error_analysis.png")

In [11]:
cap = cv2.VideoCapture(str(PATH_A))
ret, fr0 = cap.read()
if not ret:
    raise RuntimeError("Cannot read video A")
mask0 = np.ones((H, W), dtype=np.uint8) * 255
x0s, y0s, x1s, y1s = [], [], [], []
for i in range(n_used):
    if i > 0:
        ret, fr = cap.read()
        if not ret:
            break
    Hwarp = h_to_affine(W_list[i])
    m = cv2.warpAffine(mask0, Hwarp, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    ys, xs = np.where(m > 0)
    if xs.size == 0:
        continue
    x0s.append(int(xs.min())); y0s.append(int(ys.min())); x1s.append(int(xs.max())); y1s.append(int(ys.max()))
cap.release()

p_lo, p_hi = CFG_A["crop_percentiles"]
x0 = int(np.percentile(x0s, p_hi)); y0 = int(np.percentile(y0s, p_hi))
x1 = int(np.percentile(x1s, p_lo)); y1 = int(np.percentile(y1s, p_lo))
marg = int(CFG_A["crop_margin"])
x0 = int(np.clip(x0 + marg, 0, W-2)); y0 = int(np.clip(y0 + marg, 0, H-2))
x1 = int(np.clip(x1 - marg, x0+1, W-1)); y1 = int(np.clip(y1 - marg, y0+1, H-1))
crop_w = x1 - x0 + 1
crop_h = y1 - y0 + 1
print("Crop bbox:", (x0, y0, x1, y1), "crop size:", (crop_w, crop_h), "output size:", (W, H))


Crop bbox: (16, 91, 1268, 907) crop size: (1253, 817) output size: (1920, 1280)


In [12]:
cap = cv2.VideoCapture(str(PATH_A))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_vid = cv2.VideoWriter(str(OUT_A / "stabilized.mp4"), fourcc, fpsA, (W, H))

raw_mag = [float(math.hypot(float(H[0,2]), float(H[1,2]))) for H in rel_H]
order = np.argsort(-np.array(raw_mag))
sample_idxs = []
min_sep = max(10, n_used // 8)
for j in order:
    jj = int(j)
    if all(abs(jj - s) >= min_sep for s in sample_idxs):
        sample_idxs.append(jj)
    if len(sample_idxs) >= 4:
        break
sample_idxs = sorted(sample_idxs)
idx_set = set(sample_idxs)
samples = []
warping_samples = []

for i in range(n_used):
    ret, fr = cap.read()
    if not ret:
        break
    Hwarp = h_to_affine(W_list[i])
    warped = cv2.warpAffine(fr, Hwarp, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
    warped_uncropped = warped.copy()
    cropped = warped[y0:y1+1, x0:x1+1]
    resized = cv2.resize(cropped, (W, H), interpolation=cv2.INTER_LINEAR)
    out_vid.write(resized)
    if i in idx_set:
        inp_crop = fr[y0:y1+1, x0:x1+1]
        inp_crop = cv2.resize(inp_crop, (W, H), interpolation=cv2.INTER_LINEAR)
        samples.append((i, inp_crop.copy(), resized.copy()))
        warping_samples.append((fr.copy(), warped_uncropped.copy(), resized.copy()))
cap.release()
out_vid.release()

make_before_after(samples, fpsA, OUT_A / "before_after.png")

tiles = []
for a,b,c in warping_samples:
    cv2.putText(b, "WARP (uncropped)", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 3, cv2.LINE_AA)
    cv2.putText(b, "WARP (uncropped)", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 1, cv2.LINE_AA)
    tiles.append(np.concatenate([a, b, c], axis=1))
warp_grid = np.concatenate(tiles, axis=0)
cv2.imwrite(str(OUT_A / "warping_examples.png"), warp_grid)

print("Saved:", OUT_A / "stabilized.mp4")


Saved: outputs/A/stabilized.mp4


### Variant A — Error analysis, interpretation, conclusions

- **Camera trajectory**: the smoothed curve removes high‑frequency shake while keeping slow intentional motion.
- **Residual/error plot**: spikes typically correspond to challenging moments (e.g., defocus / low texture). Robust estimation (RANSAC + reinit fallback) should keep the output stable.
- **Before/after frames**: compare **input (cropped)** vs **stabilized** on the same crop region to avoid confusing border/crop effects.
- **Trade‑off**: stronger stabilization usually requires a larger crop (less field of view).


In [13]:
CFG_B = {
    "lk_max_corners": 400,
    "lk_track_frames": 200,
    "farneback_frames": 60,
    "mag_percentile": 95,
    "blur_ksize": 9,
    "noise_sigma": 10,
    "shadow_alpha": 0.65
}

## Variant B — Optical Flow, Motion Mask, Sensitivity

We visualize **LK sparse tracking**, compute **dense Farnebäck flow**, build a **motion mask** (after compensating global motion), and test **LK robustness** under blur/noise/shadow.


In [14]:
def read_frames(path, max_frames=None):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return [], 0.0
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ret, fr = cap.read()
        if not ret:
            break
        frames.append(fr)
        if max_frames is not None and len(frames) >= int(max_frames):
            break
    cap.release()
    return frames, float(fps)

framesB, fpsB = read_frames(PATH_B, max_frames=CFG_B["lk_track_frames"])
if len(framesB) < 3:
    raise RuntimeError("Video B too short")
print("Variant B frames:", len(framesB), "fps:", fpsB, "shape:", framesB[0].shape)

Variant B frames: 200 fps: 10.0 shape: (576, 768, 3)


In [15]:
def lk_overlay_image(frames, max_corners=400, track_len=20, reinit_min=120, fb_thresh=1.5, quality=0.01, min_dist=7):
    if len(frames) < 3:
        return None
    lk_params = dict(winSize=(21,21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(prev_gray, maxCorners=max_corners, qualityLevel=quality, minDistance=min_dist, blockSize=7)
    tracks = []
    if p0 is not None:
        for pt in p0.reshape(-1,2):
            tracks.append([pt])
    mid = len(frames) // 2
    snap_frame = None
    snap_tracks = None
    for i in range(1, len(frames)):
        gray = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        if len(tracks) > 0:
            p_prev = np.array([t[-1] for t in tracks], dtype=np.float32).reshape(-1,1,2)
            p_next, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, p_prev, None, **lk_params)
            if p_next is None:
                tracks = []
            else:
                p_back, st2, err2 = cv2.calcOpticalFlowPyrLK(gray, prev_gray, p_next, None, **lk_params)
                fb = np.linalg.norm(p_prev.reshape(-1,2) - p_back.reshape(-1,2), axis=1)
                good = (st.reshape(-1) == 1) & (st2.reshape(-1) == 1) & (fb < fb_thresh)
                new_tracks = []
                p_next2 = p_next.reshape(-1,2)
                for t, pt, g in zip(tracks, p_next2, good):
                    if not g:
                        continue
                    t.append(pt)
                    if len(t) > track_len:
                        t.pop(0)
                    new_tracks.append(t)
                tracks = new_tracks
        if len(tracks) < reinit_min:
            mask = np.ones_like(gray, dtype=np.uint8) * 255
            for t in tracks:
                x, y = np.round(t[-1]).astype(int)
                if 0 <= x < mask.shape[1] and 0 <= y < mask.shape[0]:
                    cv2.circle(mask, (x, y), 12, 0, -1)
            add = max(0, max_corners - len(tracks))
            if add > 0:
                p_new = cv2.goodFeaturesToTrack(gray, maxCorners=add, qualityLevel=quality, minDistance=min_dist, blockSize=7, mask=mask)
                if p_new is not None:
                    for pt in p_new.reshape(-1,2):
                        tracks.append([pt])
        if i == mid:
            snap_frame = frames[i].copy()
            snap_tracks = [t.copy() for t in tracks]
        prev_gray = gray
    if snap_frame is None:
        snap_frame = frames[-1].copy()
        snap_tracks = [t.copy() for t in tracks]
    vis = snap_frame.copy()
    for t in snap_tracks:
        if len(t) < 2:
            continue
        pts = np.round(np.array(t, dtype=np.float32)).astype(int)
        for j in range(1, len(pts)):
            a = tuple(pts[j-1])
            b = tuple(pts[j])
            cv2.line(vis, a, b, (0,255,0), 1, cv2.LINE_AA)
        cv2.circle(vis, tuple(pts[-1]), 2, (0,255,0), -1, cv2.LINE_AA)
    return vis

lk_img = lk_overlay_image(framesB, max_corners=CFG_B["lk_max_corners"])
if lk_img is None:
    raise RuntimeError("LK visualization failed")
cv2.imwrite(str(OUT_B / "lk_trajectories.png"), lk_img)
print("Saved:", OUT_B / "lk_trajectories.png")


Saved: outputs/B/lk_trajectories.png


In [16]:
def estimate_affine_curr_to_prev(prev_gray, curr_gray, max_corners=600):
    p0 = cv2.goodFeaturesToTrack(prev_gray, maxCorners=max_corners, qualityLevel=0.01, minDistance=7, blockSize=7)
    if p0 is None or len(p0) < 8:
        return np.array([[1.0,0.0,0.0],[0.0,1.0,0.0]], dtype=np.float32)
    lk_params = dict(winSize=(21,21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, p0, None, **lk_params)
    if p1 is None:
        return np.array([[1.0,0.0,0.0],[0.0,1.0,0.0]], dtype=np.float32)
    good = st.reshape(-1) == 1
    a = p0.reshape(-1,2)[good]
    b = p1.reshape(-1,2)[good]
    if a.shape[0] < 8:
        return np.array([[1.0,0.0,0.0],[0.0,1.0,0.0]], dtype=np.float32)
    M, inl = cv2.estimateAffinePartial2D(b, a, method=cv2.RANSAC, ransacReprojThreshold=3.0, maxIters=2000, confidence=0.99, refineIters=10)
    if M is None:
        return np.array([[1.0,0.0,0.0],[0.0,1.0,0.0]], dtype=np.float32)
    return M.astype(np.float32)

def farneback_residual(frames, num=60, mag_percentile=95):
    num = min(int(num), len(frames)-1)
    prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    prev = cv2.GaussianBlur(prev, (5,5), 0)
    flow_vis = None
    mask_vis = None
    mid = max(1, num // 2)
    k = np.ones((3,3), np.uint8)
    for i in range(1, num+1):
        curr = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        curr = cv2.GaussianBlur(curr, (5,5), 0)
        M = estimate_affine_curr_to_prev(prev, curr)
        curr_w = cv2.warpAffine(curr, M, (curr.shape[1], curr.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        flow = cv2.calcOpticalFlowFarneback(prev, curr_w, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
        thr = float(np.percentile(mag, mag_percentile))
        thr = max(thr, 0.8)
        mask = (mag > thr).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, k, iterations=1)
        if i == mid:
            hsv = np.zeros_like(frames[i])
            hsv[...,1] = 255
            hsv[...,0] = (ang * 180 / np.pi / 2).astype(np.uint8)
            scale = np.percentile(mag, 99)
            scale = max(scale, thr)
            hsv[...,2] = np.clip(mag / (scale + 1e-6) * 255, 0, 255).astype(np.uint8)
            flow_vis = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
            mask_vis = mask
        prev = curr
    return flow_vis, mask_vis

flow_img, mask_img = farneback_residual(framesB, num=CFG_B["farneback_frames"], mag_percentile=CFG_B["mag_percentile"])
if flow_img is None or mask_img is None:
    raise RuntimeError("Farneback residual demo failed")
cv2.imwrite(str(OUT_B / "dense_flow.png"), flow_img)
cv2.imwrite(str(OUT_B / "motion_mask.png"), mask_img)
print("Saved:", OUT_B / "dense_flow.png", OUT_B / "motion_mask.png")


Saved: outputs/B/dense_flow.png outputs/B/motion_mask.png


In [17]:
def count_components(mask, min_area=150):
    num, lab, stats, cent = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num <= 1:
        return 0
    areas = stats[1:, cv2.CC_STAT_AREA]
    return int(np.sum(areas >= int(min_area)))

def lk_survival_curve(frames, p0, T=200, fb_thresh=1.5):
    if p0 is None or len(frames) < 3:
        return None
    T = min(int(T), len(frames)-1)
    lk_params = dict(winSize=(21,21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    p = p0.copy()
    counts = [int(p.shape[0])]
    for i in range(1, T+1):
        g = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        p1, st, err = cv2.calcOpticalFlowPyrLK(prev, g, p, None, **lk_params)
        if p1 is None:
            counts.extend([0] * (T - i + 1))
            break
        p0r, st2, err2 = cv2.calcOpticalFlowPyrLK(g, prev, p1, None, **lk_params)
        fb = np.linalg.norm(p.reshape(-1,2) - p0r.reshape(-1,2), axis=1)
        good = (st.reshape(-1) == 1) & (st2.reshape(-1) == 1) & (fb < float(fb_thresh))
        if int(np.sum(good)) < 5:
            counts.append(int(np.sum(good)))
            counts.extend([int(np.sum(good))] * (T - i))
            break
        p = p1[good].reshape(-1,1,2)
        prev = g
        counts.append(int(p.shape[0]))
    return np.array(counts, dtype=np.int32)

def apply_blur(frames, ksize):
    k = int(ksize)
    if k % 2 == 0:
        k += 1
    return [cv2.GaussianBlur(f, (k,k), 0) for f in frames]

def apply_noise(frames, sigma):
    out = []
    s = float(sigma)
    for f in frames:
        n = np.random.normal(0.0, s, f.shape).astype(np.float32)
        out.append(np.clip(f.astype(np.float32) + n, 0, 255).astype(np.uint8))
    return out

def apply_shadow(frames, alpha):
    out = []
    a = float(alpha)
    for f in frames:
        out.append(np.clip(f.astype(np.float32) * a, 0, 255).astype(np.uint8))
    return out

def sensitivity_run(frames):
    prev0 = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(prev0, maxCorners=int(CFG_B["lk_max_corners"]), qualityLevel=0.01, minDistance=7, blockSize=7)
    blur_k = max(int(CFG_B["blur_ksize"]), 15)
    noise_s = max(float(CFG_B["noise_sigma"]), 20.0)
    shadow_a = min(float(CFG_B["shadow_alpha"]), 0.55)
    variants = {
        "original": frames,
        "blur": apply_blur(frames, blur_k),
        "noise": apply_noise(frames, noise_s),
        "shadow": apply_shadow(frames, shadow_a)
    }
    curves = {}
    lk_end = {}
    lk_auc = {}
    comps = {}
    T = min(250, len(frames)-1)
    for k, frs in variants.items():
        c = lk_survival_curve(frs, p0, T=T, fb_thresh=1.5)
        curves[k] = c
        if c is None:
            lk_end[k] = 0
            lk_auc[k] = 0.0
        else:
            lk_end[k] = int(c[-1])
            lk_auc[k] = float(np.mean(c / max(1.0, float(c[0]))))
        flow_img, mask_img = farneback_residual(frs, num=min(60, len(frs)-1), mag_percentile=CFG_B["mag_percentile"])
        comps[k] = count_components(mask_img, min_area=200) if mask_img is not None else 0
    return curves, lk_end, lk_auc, comps, T

curves, lk_end, lk_auc, comps, T = sensitivity_run(framesB)

labels = list(lk_end.keys())
t = np.arange(T+1)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
for k in labels:
    c = curves[k]
    if c is None:
        continue
    y = c / max(1.0, float(c[0])) * 100.0
    plt.plot(t[:len(y)], y, label=k)
plt.ylim(0, 105)
plt.title("LK survival (%)")
plt.xlabel("frame")
plt.ylabel("% of initial points")
plt.legend()

plt.subplot(1,2,2)
plt.bar(labels, [comps[k] for k in labels])
plt.title("Mask components")
plt.tight_layout()
plt.savefig(str(OUT_B / "sensitivity.png"), dpi=200)
plt.close()
print("Saved:", OUT_B / "sensitivity.png")

Saved: outputs/B/sensitivity.png


### Variant B — Interpretation, conclusions, limitations

- **LK overlay**: short trajectories should follow stable corners; failures usually come from textureless regions, occlusions, or strong illumination changes.
- **Dense flow**: after global-motion compensation, remaining flow highlights **moving objects** rather than camera motion.
- **Motion mask**: clean blobs indicate consistent object motion; many tiny components usually mean noise (mitigated by morphology + area filtering).
- **Sensitivity**: survival curves quantify how many initial LK points remain trackable over time; blur/noise/shadow should reduce survival compared to the original.
